# Load and classify new contacts from LinkedIn

Pulls first-degree connections that are **not yet in Firestore**, fetches each
full profile through Unipile, stores it in `extracted` in the LinkedIn Helper
shape, and then classifies **every stored profile that has no classification yet**
into `analysis`.

This replaces the LinkedIn Helper browser extension as the source of profiles.
Everything downstream — the prompt, the schema, the collections — is unchanged,
so records created here sit alongside the existing ~28k without any migration.

### Storing and classifying are separate steps

Phase D stores the batch it fetched. Phase E then throws that list away and asks
Firestore the only question that matters: *which documents in `extracted` have no
complete document in `analysis`?* Everything it finds is classified — this run's
profiles, profiles an interrupted earlier run stored, and profiles POSTed to
`main.py` by the browser extension.

Both phases derive their work from durable state rather than from what happened
to be in memory, so a profile can be stored and then forgotten only if Firestore
loses it. Stop the notebook anywhere and re-run it: it picks up exactly where it
left off.

### Before you run it

- **Do a dry run first.** Set `DRY_RUN = True` in the configuration cell. Phases
  A–C run for real (they read from LinkedIn), but nothing is written to
  Firestore and no Gemini calls are made.
- **This is a multi-day job for a large backlog.** Profile fetches are capped by
  `UNIPILE_MAX_PROFILE_FETCHES_PER_DAY` (default 250) because LinkedIn throttles
  and restricts accounts that read too fast. The run stops cleanly when the
  budget is spent and picks up where it left off tomorrow — Phase A recomputes
  what is already stored, so re-running is always safe.
- **Watch the first Phase E count.** It reports the whole unclassified backlog,
  not just this run's batch. If that number is large and you do not want to spend
  the Gemini budget in one sitting, cap it with `MAX_CLASSIFY`.

In [ ]:
# Load and classify new LinkedIn contacts via Unipile
import logging
import os
import time
from google.api_core import retry
import polars as pl
from dotenv import load_dotenv
from google import genai

from google.cloud import firestore
from google.genai import types
from pydantic import BaseModel

from functions import get_member_distance, join_keys
from lib.unipile import SUMMARY_KEYS, UnipileClient, to_lh_document
from lib.unipile.errors import (
    BudgetExhausted,
    ProfileIncomplete,
    ThrottleLockout,
    UnipileError,
)

load_dotenv()

# The cadence pauses for minutes at a time on purpose; log it so a long silence
# in Phase C reads as a break rather than a hang.
logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("lib.unipile").setLevel(logging.INFO)

# Initialize the Gemini client (auto-detects GOOGLE_API_KEY from environment)
client = genai.Client()

# Initialize Firestore client with specific project ID, only if file is found.
# This assignment also overrides the dev container default of
# /etc/credentials.json, which does not exist in the image.
if os.path.isfile("vk-linkedin-master-service-account.json"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
        "vk-linkedin-master-service-account.json"
    )


In [ ]:

# ── Configuration ─────────────────────────────────────────────────────────────

# Dry run: fetch profiles and report what WOULD happen, without writing to
# Firestore or calling Gemini. Use this for the first run.
DRY_RUN = False

# Cap this run below the daily budget. None means "use the whole budget".
MAX_NEW_PROFILES = None

# Cap how many profiles Phase E sends to Gemini. None means "clear the whole
# backlog". Phase E works from Firestore, so a cap only defers work — whatever
# is left over is picked up by the next run of this notebook or analysis.ipynb.
MAX_CLASSIFY = None

# Profiles with a summary shorter than this are not worth classifying.
# Same threshold as analysis.ipynb.
SUMMARY_MIN_LEN = 50

# retry policy with custom parameters
# reference https://cloud.google.com/spanner/docs/custom-timeout-and-retry?hl=en
my_retry = retry.Retry(
    initial=0.5,   # Initial delay of 0.5 seconds
    timeout=60,    # Maximum timeout of 60 seconds
    multiplier=2,  # Increase delay by 2x on each retry
)

print("Configuration ready.")
if DRY_RUN:
    print("⚠️  DRY_RUN=True — nothing will be written to Firestore or sent to Gemini.")

In [ ]:

db = firestore.Client(project="vk-linkedin", database="linkedin")

extracted_ref = db.collection("extracted")
analysis_ref = db.collection("analysis")

# The Unipile client is created without a `with` block because notebooks do not
# reliably run __exit__. The last cell closes it.
li = UnipileClient.from_env()

print(f"LinkedIn account: {li.account_id}")
print(f"Profile sections: {', '.join(li.settings.profile_sections)}")
print(f"Daily profile budget: {li.budget.remaining('profile')} remaining today")

In [ ]:
# System instructions for analyzing LinkedIn profile summaries
ANALYSIS_INSTRUCTIONS = """You classify LinkedIn profiles for a healthcare RCM technology company. The goal is to find potential buyers of Recovr by Pinnacle Services, AI-powered claim denial recovery software sold to pathology practices, independent and physician-practice-owned medical laboratories, and revenue cycle management (RCM) and medical billing companies. Hospital-owned laboratories are not a target, so the industry labels below separate laboratories by who owns them.

# Input
You receive one flattened LinkedIn profile as plain text. It is a mechanical dump, not prose: it may contain field names, quotation marks, timestamps, numeric ids, image URLs and repeated values. Ignore all of that and read only the professional content: headline, roles and employers, role descriptions, About text, skills, certifications and education.

The profile may list many roles, past and present.

# Rule 1: classify the CURRENT role
industry, function and seniority all describe the person's current position.
- Current means the role with no end date, or the most recently started role if none is open-ended. If several roles have no end date, use the one the headline describes; otherwise the most recently started.
- If the person is retired, between roles, or lists only past roles, use the most recent role.
- Use earlier roles, the About text, skills and certifications only to resolve ambiguity in the current role, never to override it. A former nurse who now leads a coding team is Operations, not Clinical.

# Rule 2: industry is the EMPLOYER'S business, not the person's job
A billing manager employed by a hospital is "Hospital". A sales representative employed by a laboratory is "Medical Lab".

industry, exactly one of:
- "RCM": revenue cycle management, medical billing, coding, CDI, HIM, denial management or collections companies, including RCM consultancies and outsourcing arms
- "Pathology": pathology practices and groups (anatomic, clinical, dermatopathology, histology), including the laboratories they operate. When both Pathology and Medical Lab fit, use Pathology.
- "Medical Lab": laboratories that test human specimens for clinical purposes and are NOT owned by a hospital: independent labs (Quest, Labcorp, regional reference, molecular, genetic and toxicology labs) and laboratories owned by a physician practice. A practice owns a lab when the profile shows it: a laboratory title (laboratory director, administrator, manager, technologist), CLIA, in-office or in-house testing, or a described laboratory. Cannabis, food, environmental, forensic and industrial testing labs are "Non-Healthcare".
- "Physician Practice": physician practices, medical groups, clinics, urgent care, ambulatory surgery centers and therapy practices with no laboratory in evidence
- "Hospital": hospitals, health systems and academic medical centers, including their laboratories, pathology departments and billing departments
- "Health IT": healthcare software, health technology and data or analytics vendors serving healthcare
- "Health Insurance Payer": insurers, health plans, managed care and other payer-side organizations
- "Pharma": pharmaceutical, biotech, medical device and laboratory instrument or reagent companies
- "Other Healthcare": healthcare-adjacent organizations not listed above, such as staffing, healthcare consulting firms, associations, research institutes, non-profits, home health, behavioral health and senior care
- "Non-Healthcare": not in healthcare

# Rule 3: function is the domain of the work
function, exactly one of:
- "Operations": revenue cycle, billing, coding, CDI, HIM, claims, denials, collections, patient access, practice or laboratory operations, quality and compliance
- "Finance": finance, accounting, controller, FP&A, CFO and other financial leadership inside an operating organization. Investment, private equity, banking, M&A and corporate development roles are "Other", not Finance.
- "IT": software, IT, data, engineering, informatics, CIO and CTO
- "Clinical": practising clinicians in a care-delivery or diagnostic role: physicians, pathologists, nurses, laboratory technologists, therapists
- "Executive": general management only: CEO, President, Managing Director, General Manager, Executive Director, board member. Never use it for an executive who leads a specific function.
- "Consulting": delivers consulting or advisory services as their role
- "Owner": use only when the person owns or founded the business and nothing indicates what they do day to day
- "Sales & Marketing": sales, business development, partnerships, marketing, account management and customer success
- "Other": none of the above

Functional executives take their domain, not "Executive": CFO is Finance; COO is Operations; CIO or CTO is IT; Chief Revenue Officer or Chief Marketing Officer is Sales & Marketing; Chief Medical Officer or Chief Nursing Officer is Clinical; Chief Revenue Cycle Officer or VP of Revenue Cycle is Operations. A General Manager of a region, segment or business unit takes the domain the headline or description emphasises, and is "Executive" only when nothing narrower applies. A founder takes the domain the headline or description shows: one who builds the product is IT, one who sells is Sales & Marketing, one who runs the company as CEO or President is Executive. A consultant at a consulting firm is Consulting, but a VP of Sales at a consulting firm is Sales & Marketing.

# Rule 4: seniority from the title
seniority, exactly one of:
- "Owner": owner, founder, co-founder, partner or principal of their own firm. This takes precedence over every other level: a founder and CEO is "Owner".
- "Executive": C-level (Chief X Officer), President, EVP, SVP, Managing Director, Executive Director, General Manager of a whole company
- "VP": Vice President, AVP, head of a function at a large organization, department Chair or Medical Director at a hospital or academic medical center, General Manager of a region or business unit
- "Director": Director, Senior Director, Associate Director, head of a team
- "Manager": Manager, Supervisor, Team Lead, Practice Administrator, Laboratory Manager
- "Staff": individual contributors: analyst, specialist, coordinator, representative, technologist, nurse, physician or consultant without a leadership title
- "Unknown": no title information at all

Board members, advisors and retired people take their most recent operating role; if none is given, use "Executive".

# Examples
- Chief Financial Officer, Sunrise Pathology Associates -> Pathology / Finance / Executive
- Founder and CEO, ClaimPath Billing Services -> RCM / Executive / Owner
- VP Coding Quality at a medical coding services company, formerly a hospital CDI nurse -> RCM / Operations / VP
- Regional Sales Director, Quest Diagnostics -> Medical Lab / Sales & Marketing / Director
- Histotechnologist, University Hospital -> Hospital / Clinical / Staff
- Practice Administrator, Coastal Dermatology -> Physician Practice / Operations / Manager
- Laboratory Administrator at a multi-physician endocrinology practice -> Medical Lab / Operations / Manager
- Chair of Pathology and Laboratory Medicine, academic medical center -> Hospital / Clinical / VP
- Controller, regional reference laboratory -> Medical Lab / Finance / Manager
- Vice President, healthcare private equity fund -> Non-Healthcare / Other / VP
- Revenue Cycle Manager, Memorial Health System -> Hospital / Operations / Manager

# Output
Return JSON with exactly the keys industry, function and seniority, using only the values listed above. Use "Other" or "Unknown" only when the profile gives no basis for a decision. Never return an empty string."""

In [ ]:
# Pydantic model for structured output
from typing import Literal

# The controlled vocabulary lives in the schema Gemini must satisfy, so an
# out-of-vocabulary label fails validation instead of landing in Firestore.
Industry = Literal[
    "RCM", "Pathology", "Medical Lab", "Physician Practice", "Hospital", "Health IT",
    "Health Insurance Payer", "Pharma", "Other Healthcare", "Non-Healthcare",
]
Function = Literal[
    "Operations", "Finance", "IT", "Clinical", "Executive", "Consulting", "Owner",
    "Sales & Marketing", "Other",
]
Seniority = Literal["Executive", "VP", "Director", "Manager", "Staff", "Owner", "Unknown"]


class ProfileAnalysis(BaseModel):
    industry: Industry
    function: Function
    seniority: Seniority


def analyze_summary(summary: str) -> dict | None:
    """
    Analyze the summary using Google Gemini with structured output.

    Args:
        summary (str): The profile summary to analyze.

    Returns:
        dict | None: Dictionary with industry, function, seniority keys, or None on
        error -- including a response outside the controlled vocabulary.
    """
    summary_preview = summary[:100] + "..." if len(summary) > 100 else summary

    # No trailing comma after the closing paren: that would make `config` a tuple
    # and the SDK would fail with "'tuple' object has no attribute 'tools'".
    # AFC is disabled because this call passes no tools; it also silences the
    # SDK's once-per-process automatic-function-calling advisory.
    config = types.GenerateContentConfig(
        system_instruction=ANALYSIS_INSTRUCTIONS,
        response_mime_type="application/json",
        response_schema=ProfileAnalysis,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    )

    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=summary,
            config=config,
        )

        # Parse and validate response via Pydantic
        result = ProfileAnalysis.model_validate_json(response.text)
        return result.model_dump()

    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        print(f"Summary: {summary_preview}")
        return None


In [ ]:
# ── Phase A: Which contacts do we already have? ───────────────────────────────
#
# A field-mask projection returns document names only, so this reads 28k ids
# without pulling 28k document bodies.
#
# Recomputing this on every run is what makes the notebook resumable: whatever
# a previous (or interrupted) run stored is skipped automatically.

print("Phase A: streaming existing document ids from 'extracted'...")

known_ids = {doc.id for doc in extracted_ref.select([]).stream()}

print(f"  already stored: {len(known_ids)}")

In [ ]:
# ── Phase B: Find connections that are not stored yet ─────────────────────────
#
# `public_identifier` is the LinkedIn slug and doubles as the Firestore document
# id, so the two sets compare directly with no mapping step.

print("Phase B: listing first-degree connections...")

candidates = []
seen = set()

for relation in li.users.iter_relations():
    slug = relation.public_identifier
    if not slug or slug in known_ids or slug in seen:
        continue
    seen.add(slug)
    candidates.append(relation)

print(f"  connections not yet stored: {len(candidates)}")

budget_left = li.budget.remaining("profile")
planned = min(len(candidates), budget_left)
if MAX_NEW_PROFILES is not None:
    planned = min(planned, MAX_NEW_PROFILES)

print(f"  daily profile budget left:  {budget_left}")
print(f"  will attempt this run:      {planned}")
if len(candidates) > planned:
    print(f"  remaining for later runs:   {len(candidates) - planned}")

In [ ]:
# ── Phase C: Fetch full profiles ──────────────────────────────────────────────
#
# This is the slow, rate-limited phase. Every fetch is delayed by a random
# interval and charged against the daily budget, because profile reads are what
# LinkedIn throttles hardest.
#
# A profile whose sections LinkedIn withheld is NOT stored. Withheld sections
# arrive empty with HTTP 200, so storing one would cache a Gemini classification
# built from a nearly empty summary — silently, and permanently. Those slugs are
# simply retried on a later run. A merely short section is fine: LinkedIn
# collapses grouped roles, so exact counts often do not match.
#
# Retries bound one slug; the lockout bounds the run. Once
# UNIPILE_MAX_CONSECUTIVE_THROTTLED profiles in a row come back withheld, the
# account is throttled rather than the profile, and continuing would burn the
# whole day's budget at the 8x pace for nothing.

s = li.settings

print(f"Phase C: fetching up to {planned} profiles...")
print(f"  gap between fetches: {s.min_delay_seconds:.0f}–{s.max_delay_seconds:.0f}s, skewed low")
if s.long_pause_every:
    print(
        f"  long break: every ~{s.long_pause_every} calls, "
        f"{s.long_pause_min_seconds:.0f}–{s.long_pause_max_seconds:.0f}s"
    )
else:
    print("  long break: disabled")
print(f"  retries when LinkedIn withholds sections: {s.throttle_retries}")
if s.max_consecutive_throttled:
    print(f"  stop after {s.max_consecutive_throttled} throttled profiles in a row")
else:
    print("  stop on sustained throttling: disabled")

profiles = []
incomplete = []
failed = []
locked_out = False

# Determine the progress step for periodic status updates
progress_step = max(1, planned // 10)

for idx, relation in enumerate(candidates[:planned], 1):
    slug = relation.public_identifier
    try:
        profile = li.users.get_profile(slug, require_complete=True)
    except BudgetExhausted as exc:
        print(f"  daily budget spent after {idx - 1} profiles: {exc.title}")
        break
    except ThrottleLockout as exc:
        # LinkedIn is throttling the account, not this profile.
        locked_out = True
        incomplete.append((slug, exc.title))
        print(f"\n  STOPPED after {idx} profiles: {exc.title}")
        print(f"  {exc.detail}")
        break
    except ProfileIncomplete as exc:
        # Already retried behind a widening pause; LinkedIn is still withholding.
        incomplete.append((slug, exc.title))
        continue
    except UnipileError as exc:
        failed.append((slug, exc.type or exc.title))
        continue

    profiles.append(profile)

    if idx % progress_step == 0 or idx == planned:
        ts = time.strftime("%H:%M:%S")
        print(f"{ts} | fetched {idx}/{planned} ({idx * 100 // planned}%)  (complete: {len(profiles)})")

print("\nPhase C results:")
print(f"  complete profiles:  {len(profiles)}")
print(f"  withheld (retry later): {len(incomplete)}")
print(f"  errors:             {len(failed)}")
for slug, reason in incomplete[:5]:
    print(f"    withheld:  {slug} → {reason}")
for slug, reason in failed[:5]:
    print(f"    error:     {slug} → {reason}")
if locked_out:
    print("\n⚠️  Fetching stopped early. Phases D and E still run on what was")
    print("    collected; re-run the notebook later to continue fetching.")


In [ ]:
# ── Phase D: Map to the LinkedIn Helper shape and store ───────────────────────

print(f"Phase D: preparing {len(profiles)} documents...")

new_docs = []

for profile in profiles:
    document = to_lh_document(profile)
    document["summary"] = join_keys(document, SUMMARY_KEYS)
    doc_id = document["id"]

    if len(document["summary"]) <= SUMMARY_MIN_LEN:
        continue

    # Resolved by the Firestore server, so it is immune to client clock skew.
    document["created_at"] = firestore.SERVER_TIMESTAMP

    # save to Firestore
    extracted_ref.document(doc_id).set(document)

    new_docs.append(
        {
            "doc_id": doc_id,
            "summary": document["summary"].replace("\n", " "),
            "profileUrl": document.get("profileUrl", "") or "",
            "memberDistance": document.get("memberDistance", 0),
        }
    )

print(f"  {'would store' if DRY_RUN else 'stored'}: {len(new_docs)} documents in 'extracted'")
print(f"  skipped (summary shorter than {SUMMARY_MIN_LEN} chars): {len(profiles) - len(new_docs)}")


In [ ]:

if new_docs:
    preview = (
        pl.DataFrame(new_docs)
        .with_columns(
            pl.col("summary").str.slice(0, 90).alias("summary_preview"),
            pl.col("summary").str.len_chars().alias("summary_length"),
        )
        .select(["doc_id", "memberDistance", "summary_length", "summary_preview"])
    )
    print()
    print(preview.head(10))

In [ ]:
# ── Phase E: Classify every stored profile that has no classification ─────────
print("Phase E: finding stored profiles without a classification...")

CATEGORY_FIELDS = ["industry", "function", "seniority"]

# A doc counts as classified only when all three categories are non-empty; a
# half-written analysis doc from a failed run must not mask the missing work.
classified_ids = {
    doc.id
    for doc in analysis_ref.select(CATEGORY_FIELDS).stream()
    if all((doc.to_dict() or {}).get(field, "") for field in CATEGORY_FIELDS)
}

print(f"  already classified: {len(classified_ids)}")

pending = []
extracted_total = 0
too_short = 0

for doc in extracted_ref.select(
    ["summary", "profileUrl", "lhId", "memberDistance"]
).stream():
    extracted_total += 1

    if doc.id in classified_ids:
        continue

    data = doc.to_dict() or {}
    summary = (data.get("summary") or "").replace("\n", " ")

    if len(summary) <= SUMMARY_MIN_LEN:
        too_short += 1
        continue

    pending.append(
        {
            "doc_id": doc.id,
            "summary": summary,
            "profileUrl": data.get("profileUrl", "") or "",
            "lh_id": str(data.get("lhId", "") or ""),
            "memberDistance": get_member_distance(data),
        }
    )

queue = pending if MAX_CLASSIFY is None else pending[:MAX_CLASSIFY]

print(f"  stored in 'extracted':      {extracted_total}")
print(f"  awaiting classification:    {len(pending)}")
print(f"  unclassifiable (summary shorter than {SUMMARY_MIN_LEN} chars): {too_short}")
print(f"  will classify this run:     {len(queue)}")
if len(pending) > len(queue):
    print(f"  deferred by MAX_CLASSIFY:   {len(pending) - len(queue)}")

In [ ]:
# ── Phase E (continued): send the queue to Gemini ─────────────────────────────
#
# Each document is written the moment it is classified, so an interruption keeps
# everything classified so far. The unwritten remainder is simply still missing
# from `analysis`, which is exactly the condition the cell above searches for.

print(f"Phase E: classifying {len(queue)} documents...")

classified = 0
classify_failed = 0

if DRY_RUN:
    print("  DRY_RUN=True — skipping Gemini.")
else:
    for idx, row in enumerate(queue, 1):
        try:
            analysis = analyze_summary(row["summary"])

            if analysis is not None and all(
                k in analysis for k in ("industry", "function", "seniority")
            ):
                analysis_ref.document(row["doc_id"]).set(
                    {
                        "profileUrl": row["profileUrl"],
                        "lh_id": row["lh_id"],
                        "industry": analysis["industry"],
                        "function": analysis["function"],
                        "seniority": analysis["seniority"],
                        "summary": row["summary"],
                        "memberDistance": row["memberDistance"],
                    },
                    retry=my_retry,
                )
                classified += 1
            else:
                classify_failed += 1
                time.sleep(1)

        except Exception as e:
            print(f"ERROR [{row['doc_id']}]: {e}")
            classify_failed += 1
            time.sleep(1)
            continue

        if idx % 100 == 0 or idx == len(queue):
            ts = time.strftime("%H:%M:%S")
            print(f"{ts} | Analyzed {idx}/{len(queue)}")

print(f"  classified: {classified},  failed: {classify_failed}")

In [ ]:
# ── Phase F: Summary ──────────────────────────────────────────────────────────

# Anything not stored is still a candidate next run, whether it was withheld,
# errored or never reached because the run stopped early.
remaining_candidates = len(candidates) - len(new_docs)
still_unclassified = len(pending) - classified

print("Run summary")
print(f"  connections not yet stored at start: {len(candidates)}")
print(f"  profiles fetched:                    {len(profiles) + len(incomplete)}")
print(f"  stored in 'extracted':               {len(new_docs)}{' (dry run)' if DRY_RUN else ''}")
print(f"  unclassified backlog found:          {len(pending)}")
print(f"  classified into 'analysis':          {classified}")
print(f"  still unclassified:                  {still_unclassified}")
print(f"  withheld, will retry next run:       {len(incomplete)}")
print(f"  still to fetch on later runs:        {remaining_candidates}")
print()
print(f"  profile budget left today:           {li.budget.remaining('profile')}")

if locked_out:
    print(
        "\n⚠️  Fetching stopped on sustained throttling. Wait several hours before"
        "\n    the next run — resuming immediately will just trip it again."
    )
elif remaining_candidates:
    print("\nRe-run this notebook tomorrow to continue; Phase A skips whatever is stored.")
if still_unclassified:
    print("Re-run Phase E (or analysis.ipynb) to classify the rest — nothing is lost.")
if DRY_RUN:
    print("\n⚠️  This was a dry run. Set DRY_RUN = False to write for real.")

li.close()


In [ ]:

# Run the collection-to-csv conversion
import subprocess
result = subprocess.run(["python3", "collection-tocsv.py"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
